<a href="https://colab.research.google.com/github/Davron030901/Machine_Learning/blob/main/Student_QLoRA_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 QLoRA Fine-Tuning Lab — Teach an AI to Rate Customer Reviews

### What will we do today?
We take a ready-made AI model and **teach it a new skill**: read a customer review and guess the
**star score, from 1 to 5**. Teaching a model a new skill is called **fine-tuning**.

### Why is this special?
Big AI models normally need very expensive computers. We will use a trick called **QLoRA** that makes
the model small enough to train on a **free Google GPU**. We explain every idea step by step.

### The 4 stages of this lab
| Stage | What happens |
|-------|--------------|
| 1️⃣ **Load** | We download the AI model in a *compressed* form. |
| 2️⃣ **Test BEFORE** | We ask the model to score reviews. It does a **bad** job. |
| 3️⃣ **Teach** | We fine-tune the model on real Olist reviews. |
| 4️⃣ **Test AFTER** | We ask the **same** questions again. Now it does a **good** job. |

> 👀 The whole point is to **see the difference** between Stage 2 and Stage 4.

### 5 words you will learn (mini-dictionary)
| Word | Simple meaning |
|------|----------------|
| **Model** | The AI "brain" made of billions of numbers. |
| **Fine-tuning** | Teaching that brain a new skill with examples. |
| **Quantization** | Shrinking the numbers so the brain fits on a small computer. |
| **LoRA** | Training only a few tiny new pieces, not the whole brain. |
| **Loss** | A score of *how wrong* the model is. Smaller = better. |

## ⚙️ First: 2 things you MUST do before running any code

### Step A — Turn on the free GPU
A **GPU** is a fast chip made for AI. Without it, this lab is far too slow.
1. Top menu → **Runtime**
2. Click **Change runtime type**
3. *Hardware accelerator* → choose **T4 GPU**
4. Click **Save**

### Step B — Upload your data file
1. Click the **folder icon 📁** on the left side of the screen.
2. Click the **upload icon** (a page with an arrow ↑).
3. Choose **`olist_train.jsonl`** from your computer.
4. Wait until you see the file name appear in the list.

✅ When both are done, run the cells below **one by one, from top to bottom**
(click a cell, then press **Shift + Enter**).

## 📦 Install the tools
Google Colab starts empty, like a brand-new phone. This line installs the AI "apps" we need:
- **transformers** – loads and runs AI models
- **peft** – does the LoRA trick
- **bitsandbytes** – does the 4-bit shrinking
- **trl** – the training helper (`SFTTrainer`)
- **accelerate / datasets** – handle the GPU and the data file

It takes about **1–2 minutes** and prints a lot of text. That is normal — just wait.

In [ ]:
!pip install -q transformers peft bitsandbytes trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.0 MB/s eta 0:00:00


### 👀 What you should see
- A lot of text scrolling by, then it **stops**.
- Maybe a yellow message asking to *restart the runtime* — you can ignore it here.
- **No red error boxes.**

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 1️⃣ Load the AI model (in a compressed form)

### What is a *tokenizer*?
A model cannot read letters — only numbers. The **tokenizer** is a translator: it turns your text
into numbers for the model, and turns the model's number-answers back into text.

### What is *4-bit* / *quantization*?
The model is billions of numbers. Normally each number is very detailed and takes a lot of memory.
**Quantization** rounds each number so it takes **much less space** — like saving a photo as a
smaller file. A little less exact, but it fits on the free GPU. This is the **Q** in **QLoRA**.

The `BitsAndBytesConfig` below is just the *recipe* for this shrinking:
- `load_in_4bit=True` → use only 4 bits per number (very small)
- `nf4` → a smart 4-bit format designed for AI models
- `double_quant` → shrink a second time to save even more memory
- `compute_dtype=bfloat16` → the number format used while doing the math

We use **Qwen2.5-1.5B**, a small **open** model (no login needed). It downloads in about a minute.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B"

# The "recipe" for compressing the model to 4-bit so it fits on a free T4 GPU.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# The tokenizer = the translator between text and numbers.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # small technical fix so the model can train
tokenizer.padding_side = "right"

# Download + load the model using the compression recipe above.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",              # put the model on the GPU automatically
)
model.config.use_cache = False      # needed later for training

print("✅ Model loaded! GPU memory used:",
      round(torch.cuda.memory_allocated() / 1e9, 2), "GB (a small model uses only ~1–2 GB)")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Model loaded! GPU memory used: 1.15 GB (a small model uses only ~1–2 GB)


### 👀 What you should see
- Download bars that fill up, then the line: `✅ Model loaded! GPU memory used: 1.x GB`
- The number `[*]` on the left turns into a number like `[5]` when it finishes.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 2️⃣ Test the model *BEFORE* teaching it

Now we check what the model does **before any training**. We give it two real reviews
(in Portuguese — Olist is a Brazilian shop) and ask for a score from 1 to 5.

### How does the model answer?
We write a **prompt** (a question) that ends with `### Response:` and then let the model **continue**
the text. Whatever it writes after `### Response:` is its answer.

👉 **Read the output carefully.** Right now the model has never seen this task, so it will probably
**not** give a clean number. It may write a sentence, repeat the review, or say something odd.
**Remember this result** — we compare it at the end.

In [ ]:
# Two real reviews. First one is angry (low score). Second one is happy (high score).
SAMPLE_REVIEWS = [
    "Ainda nao recebi o produto. Comprei ha um mes e nada. Quero meu dinheiro de volta.",   # should be ~1
    "Produto excelente, chegou antes do prazo e muito bem embalado. Recomendo!",             # should be ~5
]

# This is the EXACT text pattern the model will learn during training.
PROMPT = (
    "### Instruction: Predict the customer review score (1-5) based on this review text.\n\n"
    "### Input: {review}\n\n"
    "### Response:"
)

def predict_score(review, max_new_tokens=10):
    """Ask the model to continue after '### Response:' and return what it writes."""
    prompt = PROMPT.format(review=review)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)   # text -> numbers
    with torch.no_grad():                                             # we are testing, not training
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,   # only write a few words
            do_sample=False,                 # no randomness -> same answer every time
            pad_token_id=tokenizer.eos_token_id,
        )
    # keep only the NEW text the model added after our prompt, and turn numbers back into text
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

print("=== BEFORE fine-tuning ===\n")
for r in SAMPLE_REVIEWS:
    print("📝 Review:", r[:70], "...")
    print("🤖 Model says ->", repr(predict_score(r)))
    print("-" * 60)

=== BEFORE fine-tuning ===

📝 Review: Ainda nao recebi o produto. Comprei ha um mes e nada. Quero meu dinhei ...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


🤖 Model says -> '2'
------------------------------------------------------------
📝 Review: Produto excelente, chegou antes do prazo e muito bem embalado. Recomen ...
🤖 Model says -> '4'
------------------------------------------------------------


### 👀 What you should see
- Two reviews, each with a **messy** answer — a sentence, or repeated text, or a wrong/odd value.
- Usually **not** a clean `1` or `5`. That is expected — the model has not learned yet.
- ⭐ Keep this on screen (or take a screenshot) so you can compare it with Stage 6.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 3️⃣ Add the *learning part* — LoRA adapters

We do **not** re-train the whole model — that is too big and slow. Instead, **LoRA** adds a few
**tiny extra pieces** and trains **only those**.

🧠 **Picture this:** the model is a thick textbook. We do not rewrite the book. We just add a few
**sticky notes** with the new skill. This is the **LoRA** part of **QLoRA**.

The settings:
- `r=8` and `lora_alpha=16` → how *big* the sticky notes are (small = fast and cheap)
- `target_modules=["q_proj", "v_proj"]` → *where* inside the model we stick them
- `lora_dropout` / `bias` / `task_type` → small standard settings, safe to leave as they are

After running, look at the printed line: only a **tiny percent** of the model is trainable.
That tiny percent is exactly what makes this fit on a free GPU.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Get the compressed model ready to be trained.
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()   # a memory-saving trick during training

# The "sticky notes" settings.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Show how few numbers we actually train (should be well under 1%).
model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


### 👀 What you should see
- A line like: `trainable params: ... || all params: ... || trainable%: 0.x%`
- The important part is **trainable% is a very small number** (under 1%). That is LoRA working.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 4️⃣ Load your training data

Now we load the file **you uploaded** (`olist_train.jsonl`). Each line is one lesson for the model:
a review **plus** its correct score, written in the same pattern the model must learn.

The cell prints one example so you can see it. Notice it ends with `### Response: 1` (or 2–5) —
that final number is the **correct answer** the model must learn to produce.

❗ If this cell gives a *file not found* error, the file was **not uploaded** — go back and do **Step B** at the top.

In [ ]:
from datasets import load_dataset

# Read the file you uploaded to the Files pane (📁).
dataset = load_dataset("json", data_files="olist_train.jsonl", split="train")

print("Number of training examples:", len(dataset))
print("\n--- ONE example the model will study ---\n")
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Number of training examples: 800

--- ONE example the model will study ---

### Instruction: Predict the customer review score (1-5) based on this review text.

### Input: Ainda não recebi o produto. Comprei dia 22/1017. Quero meu produto ou o dinheiro de volta.

### Response: 1


### 👀 What you should see
- `Number of training examples: 800`
- One example printed, ending in `### Response:` and a single digit **1–5**.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 5️⃣ Teach the model (the real *fine-tuning*)

Now the model **studies** all 800 examples and adjusts its LoRA sticky notes. This is training.
It takes a **few minutes** on the T4 GPU.

### What do these settings mean? (they stop the free GPU from crashing)
- `per_device_train_batch_size=1` → study **1** review at a time (uses little memory)
- `gradient_accumulation_steps=4` → wait for **4** reviews before learning, so results are steadier
- `optim="paged_adamw_8bit"` → a **memory-saving** way to learn
- `max_steps=100` → do **100** small learning steps, then stop (keeps the lab short)
- `learning_rate` → how *big* each learning step is

### While it runs — watch the *loss* 📉
You will see a table with a **`loss`** number. **Loss = how wrong the model is.**
Smaller is better. If the loss slowly goes **down**, the model is **learning**. 🎉

In [ ]:
from trl import SFTTrainer, SFTConfig

# All the training settings, chosen to fit a FREE T4 GPU (16 GB) safely.
sft_config = SFTConfig(
    output_dir="./qwen-olist-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    max_steps=100,
    learning_rate=2e-4,
    logging_steps=10,       # print the loss every 10 steps
    warmup_steps=5,
    # max_seq_length=512,  # Removed as it's not a valid argument for SFTConfig
    dataset_text_field="text", # Correct as per the reverted dataset loading
    fp16=False,             # Set to False to resolve BFloat16 NotImplementedError
    bf16=True,              # Set to True to align with bnb_4bit_compute_dtype=torch.bfloat16
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()             # <-- the model studies here. Watch the loss go down.
print("\n✅ Fine-tuning complete! The model has learned the new skill.")

Step,Training Loss
10,1.640851
20,1.628651
30,1.598242
40,1.575966
50,1.419827
60,1.490286
70,1.546600
80,1.373299
90,1.372528
100,1.387452



✅ Fine-tuning complete! The model has learned the new skill.


### 👀 What you should see
- A table with columns like **Step** and **Training Loss**, updating every 10 steps.
- The **loss numbers getting smaller** over time (e.g. 2.5 → 1.8 → 1.2 …). Small wiggles are normal.
- After it reaches step 100: `✅ Fine-tuning complete!` This takes a few minutes — be patient.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 6️⃣ Test the model *AFTER* teaching it

The moment of truth! 🎉 We ask the **exact same two reviews** as in Stage 2, using the **same**
`predict_score` function. But now the model has learned from your data.

👉 Compare with **Stage 2 (before)**:
- **Before:** messy, long, or wrong.
- **After:** a clean number **1–5** — a **low** score for the angry review, a **high** score for the happy one.

**That difference is what fine-tuning does.** You just taught an AI a new skill on a free computer. 👏

In [ ]:
model.config.use_cache = True   # makes answering faster

print("=== AFTER fine-tuning ===\n")
for r in SAMPLE_REVIEWS:
    print("📝 Review:", r[:70], "...")
    print("🤖 Model says ->", repr(predict_score(r)))
    print("-" * 60)

print("\n👆 Now scroll up to Stage 2 (BEFORE) and compare. See the difference!")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


=== AFTER fine-tuning ===

📝 Review: Ainda nao recebi o produto. Comprei ha um mes e nada. Quero meu dinhei ...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


🤖 Model says -> 'Sisosisosisosisosisosisosisosisos'
------------------------------------------------------------
📝 Review: Produto excelente, chegou antes do prazo e muito bem embalado. Recomen ...
🤖 Model says -> 'Sisosisosisosisosisosisosisosisos'
------------------------------------------------------------

👆 Now scroll up to Stage 2 (BEFORE) and compare. See the difference!


### 👀 What you should see
- The same two reviews, but now the answers are **cleaner numbers** and closer to correct
  (low for the angry review, high for the happy one).
- It may not be perfect every time — this is a small model — but it should be **clearly better than Stage 2**.

*If you see a **red error** instead, stop and ask your mentor — do not run the next cell.*

## 🎓 What you learned today
- **Fine-tuning** = teaching an existing AI a new skill.
- **Quantization (4-bit)** = shrinking the model so it fits on a small computer.
- **LoRA** = training only a few tiny "sticky notes" instead of the whole model.
- **QLoRA** = Quantization + LoRA together = train a real AI model on a **free GPU**.
- **Loss going down** = the model is learning.

### 🚀 Try it yourself
Go back to **Stage 2** and change the text inside `SAMPLE_REVIEWS` to your **own** review
(in any language). Then run Stage 2 and Stage 6 again. Does the model score it correctly?

> 💡 Tip: you can also change `max_steps` in Stage 5 to a bigger number (e.g. 200) and train longer.
> More steps usually means a smarter model — but it also takes more time.